## Exploring predictions

In [1]:
import sys
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt
import scipy
import methods

import rasterio
from rasterio.windows import Window
from rasterio.transform import Affine


In [2]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")

# tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on
# print(tf.config.list_physical_devices('GPU'))


python version = 3.10.10 | packaged by conda-forge | (main, Mar 24 2023, 20:12:31) [Clang 14.0.6 ]
numpy version = 1.23.2


In [3]:
directory_paths = utils.get_directories()
SAVE_MODEL_DIR = directory_paths["save_model_dir"]
DATA_DIR = directory_paths["data_dir"]
FIGURE_DIR = directory_paths["figures_dir"]
PREDICTIONS_DIR = directory_paths["predictions_dir"]


## Re-save HII files to uint8 and scaled by 100 smaller values

In [ ]:
assert False


In [ ]:
for year in (2015, 2016, 2017, 2018, 2019, 2020):
    # for year in (2018, 2019, 2020):

    filename = DATA_DIR + "archive/hii_" + str(year) + "-01-01.tif"
    with rasterio.open(filename) as orig_tiff:
        lon_w, lat_n = orig_tiff.xy(0, 0, offset="ul")
        lon_e, lat_s = orig_tiff.xy(orig_tiff.shape[0], orig_tiff.shape[1], offset="ul")
        print(lat_s, lat_n, lon_w, lon_e)
        hfi = orig_tiff.read(1)
        meta_data = orig_tiff.meta.copy()

    NO_DATA = 255
    hfi = np.asarray(
        np.where(hfi != -32768, np.round(hfi / 6400.0 * 100), NO_DATA), dtype="uint8"
    )
    meta_data.update(
        {"dtype": hfi.dtype, "nodata": NO_DATA, "compress": "lzw"},
    )
    print(meta_data)
    print(np.min(hfi), np.max(hfi))

    print("   saving the tif file for " + str(year) + "...")
    with rasterio.open(
        "/Users/eabarnes/Documents/hii_" + str(year) + "-01-01_uint8.tif",
        "w",
        **meta_data
    ) as dst:
        dst.write(hfi, 1)

## Make coastal buffer file

In [ ]:
assert False


In [ ]:
filename = DATA_DIR + "hii_2020-01-01_uint8.tif"
with rasterio.open(filename) as orig_tiff:
    lon_w, lat_n = orig_tiff.xy(0, 0, offset="ul")
    lon_e, lat_s = orig_tiff.xy(orig_tiff.shape[0], orig_tiff.shape[1], offset="ul")

    print(lat_s, lat_n, lon_w, lon_e)

    mask = orig_tiff.read_masks(1)
    meta_data = orig_tiff.meta.copy()

In [ ]:
# blend the coastal areas and count those as "non-ocean" areas.
hfi_coastal = scipy.ndimage.gaussian_filter(np.asarray(mask, "float32"), 3, mode="wrap")
hfi_coastal = np.where(hfi_coastal != 0, 1.0, 0.0)
hfi_coastal = np.asarray(hfi_coastal, dtype="uint8")

meta_data.update(
    {"dtype": hfi_coastal.dtype, "nodata": None, "compress": "lzw"},
)

print("   saving the tif file ...")
with rasterio.open(
    DATA_DIR + "hii_coastal_buffer_mask.tif", "w", **meta_data
) as dst:
    dst.write(hfi_coastal, 1)

## Make land mask only for 2020

In [ ]:
assert False

In [4]:
filename = DATA_DIR + "hii_2020-01-01_uint8.tif"
with rasterio.open(filename) as orig_tiff:
    lon_w, lat_n = orig_tiff.xy(0, 0, offset="ul")
    lon_e, lat_s = orig_tiff.xy(orig_tiff.shape[0], orig_tiff.shape[1], offset="ul")

    print(lat_s, lat_n, lon_w, lon_e)

    mask = orig_tiff.read_masks(1)
    meta_data = orig_tiff.meta.copy()

-58.00062463446102 84.00146221801646 -180.00082337073326 180.00082337073326


In [6]:
# blend the coastal areas and count those as "non-ocean" areas.
# hfi_coastal = scipy.ndimage.gaussian_filter(np.asarray(mask, "float32"), 3, mode="wrap")
hfi_coastal = np.where(mask != 0, 1.0, 0.0)
hfi_coastal = np.asarray(hfi_coastal, dtype="uint8")

meta_data.update(
    {"dtype": hfi_coastal.dtype, "nodata": None, "compress": "lzw"},
)

print("   saving the tif file ...")
with rasterio.open(
    DATA_DIR + "hii_land_mask.tif", "w", **meta_data
) as dst:
    dst.write(hfi_coastal, 1)

   saving the tif file ...
